# NSL-KDD Intrusion Detection with XAI and LLMs

In [ ]:
!pip install shap lime -q

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from lime import lime_tabular
from openai import OpenAI

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


## Configuration

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.30
SHAP_SAMPLE_SIZE = 200

DATASET_PATH = (
    "/content/sample_data/"
    "KDDTrain+_20Percent.txt"
)


## Dataset Loading

In [ ]:
df = pd.read_csv(
    DATASET_PATH,
    header=None
)

print(df.shape)

df.head()


## Train/Test Split

In [ ]:
X = df.iloc[:, :-2]
y = df.iloc[:, -2]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


## Random Forest Training

In [ ]:
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")

print(
    classification_report(
        y_test,
        y_pred
    )
)


## SHAP Explainability

In [ ]:
sample_idx = np.random.choice(
    X_test.index,
    size=min(
        SHAP_SAMPLE_SIZE,
        len(X_test)
    ),
    replace=False
)

X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(
    rf_model
)

shap_values = explainer.shap_values(
    X_sample
)

shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar"
)


## Prompt Construction

In [ ]:
prompt = f"""
You are an expert in Explainable AI
and Cybersecurity.

Analyze the trained Random Forest model
and explain its behavior.
"""
